In [20]:
from google.colab import auth
auth.authenticate_user()

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#통합된 데이터 사용

import pandas as pd
df = pd.read_csv('데이터 경로')
print(df.head())

        site                      title  \
0  nate_pann       1살많은 연상녀...(공무원준비하는)   
1  nate_pann                 소음인에 대해...   
2  nate_pann  [부부이야기] 아내 건강의 반은 남편 몫일까?   
3  nate_pann               버릇고쳐 살아야죠...   
4  nate_pann           한국도자기 그릇세트를 사고..   

                                             content  \
0  2005년말에 만나서 제가 좋아해서 사귀자고 했고 어렵게 어렵게 사귀었습니다. 20...   
1  1. 체형의 특징 체 형 비대신소 상징 동물 나귀 치밀하고 잔재주 있음 인물평,성정...   
2  아내 나이 40줄 넘더니 점점 몸이 안좋다는 말이 자주 나온다.? 노안이 온다는둥,...   
3  니이랑 저랑 정말 똑같은 입장인것 같네여.. 제 남편은 남들한테조차도 제 얘기만 꺼...   
4  결혼 15년차입니다.. 6개월전...결혼 기념일 선물로..롯데백화점 한국도자기 코너...   

                                        cleaned_text  
0  2005년말에 만나서 제가 좋아해서 사귀자고 했고 어렵게 어렵게 사귀었습니다. 20...  
1  1. 체형의 특징 체 형 비대신소 상징 동물 나귀 치밀하고 잔재주 있음 인물평,성정...  
2  아내 나이 40줄 넘더니 점점 몸이 안좋다는 말이 자주 나온다.? 노안이 온다는둥,...  
3  니이랑 저랑 정말 똑같은 입장인것 같네여.. 제 남편은 남들한테조차도 제 얘기만 꺼...  
4  결혼 15년차입니다.. 6개월전...결혼 기념일 선물로..롯데백화점 한국도자기 코너...  


In [24]:
import torch
from sentence_transformers import SentenceTransformer

# GPU 사용 가능 여부 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 모델 로드
model = SentenceTransformer("jhgan/ko-sroberta-multitask",device=device)

# content 컬럼을 리스트로 변환
texts = df['cleaned_text'].astype(str).tolist()

# 임베딩 생성
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

# dataframe에 vector 컬럼 추가
df['vector'] = embeddings.tolist()

# 확인
df.head()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

,site,title,content,cleaned_text,vector
0,nate_pann,1살많은 연상녀...(공무원준비하는),2005년말에 만나서 제가 좋아해서 사귀자고 했고 어렵게 어렵게 사귀었습니다. 20...,2005년말에 만나서 제가 좋아해서 사귀자고 했고 어렵게 어렵게 사귀었습니다. 20...,"[-0.4270292818546295, -0.3758968114852905, -0...."
1,nate_pann,소음인에 대해...,"1. 체형의 특징 체 형 비대신소 상징 동물 나귀 치밀하고 잔재주 있음 인물평,성정...","1. 체형의 특징 체 형 비대신소 상징 동물 나귀 치밀하고 잔재주 있음 인물평,성정...","[-0.24646008014678955, -0.10394762456417084, -..."
2,nate_pann,[부부이야기] 아내 건강의 반은 남편 몫일까?,"아내 나이 40줄 넘더니 점점 몸이 안좋다는 말이 자주 나온다.? 노안이 온다는둥,...","아내 나이 40줄 넘더니 점점 몸이 안좋다는 말이 자주 나온다.? 노안이 온다는둥,...","[-0.5623100996017456, -0.2525443434715271, -0...."
3,nate_pann,버릇고쳐 살아야죠...,니이랑 저랑 정말 똑같은 입장인것 같네여.. 제 남편은 남들한테조차도 제 얘기만 꺼...,니이랑 저랑 정말 똑같은 입장인것 같네여.. 제 남편은 남들한테조차도 제 얘기만 꺼...,"[-0.24757671356201172, -0.249628484249115, 0.0..."
4,nate_pann,한국도자기 그릇세트를 사고..,결혼 15년차입니다.. 6개월전...결혼 기념일 선물로..롯데백화점 한국도자기 코너...,결혼 15년차입니다.. 6개월전...결혼 기념일 선물로..롯데백화점 한국도자기 코너...,"[-0.3157845735549927, -0.24885505437850952, -0..."


In [25]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import os
import re

output_dir = "/content"

base_name = "Ko-SBERT_embedding_data"
extension = ".csv"

existing_files = os.listdir(output_dir)

pattern = re.compile(rf"^{base_name}_(\d+){extension}$")

numbers = []

for filename in existing_files:
    match = pattern.match(filename)
    if match:
        numbers.append(int(match.group(1)))

next_number = max(numbers) + 1 if numbers else 1

df_filename = os.path.join(
    output_dir,
    f"{base_name}_{next_number:03d}{extension}"
)

df.to_csv(df_filename, index=False, encoding="utf-8-sig")
print(f"DataFrame saved to {df_filename}")

drive_service = build('drive', 'v3')

if os.path.exists(df_filename):
    file_metadata = {
        'name': os.path.basename(df_filename)
    }

    media = MediaFileUpload(
        df_filename,
        mimetype='text/csv',
        resumable=True
    )

    uploaded_file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id'
    ).execute()

    print(
        f"Uploaded '{os.path.basename(df_filename)}' "
        f"File ID: {uploaded_file.get('id')}"
    )

else:
    print(f"Warning: CSV file not found for upload - {df_filename}")

print("CSV file has been processed and uploaded to Google Drive.")

DataFrame saved to /content/Ko-SBERT_embedding_data_005.csv


Uploaded 'Ko-SBERT_embedding_data_005.csv' File ID: 1_qca1wtpVTTcZ1s6OV2mWTXUQOvjHJT3
CSV file has been processed and uploaded to Google Drive.


In [26]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

output_dir = "/content"

base_name = "similarity_comparison_result"
extension = ".txt"

existing_files = os.listdir(output_dir)

pattern = re.compile(rf"^{base_name}_(\d+){extension}$")

numbers = []

for filename in existing_files:
    match = pattern.match(filename)
    if match:
        numbers.append(int(match.group(1)))

next_number = max(numbers) + 1 if numbers else 1

txt_filename = os.path.join(
    output_dir,
    f"{base_name}_{next_number:03d}{extension}"
)

embedding_col = "vector"

min_k = 2
max_k = 10

df_valid = df.copy()
df_valid = df_valid.dropna(subset=[embedding_col]).reset_index(drop=True)

def convert_vector(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)

    if isinstance(x, list):
        return np.array(x, dtype=float)

    if isinstance(x, str):
        x = x.strip()
        x = x.replace("\n", " ")
        x = x.strip("[]")
        return np.array([float(v) for v in x.replace(",", " ").split()], dtype=float)

    return np.nan

df_valid[embedding_col] = df_valid[embedding_col].apply(convert_vector)

df_valid = df_valid[
    df_valid[embedding_col].apply(lambda x: isinstance(x, np.ndarray))
].reset_index(drop=True)

embeddings = np.vstack(df_valid[embedding_col].values)

print("사용 가능한 데이터 수:", len(df_valid))
print("임베딩 차원:", embeddings.shape[1])

similarity_matrix = cosine_similarity(embeddings)
np.fill_diagonal(similarity_matrix, np.nan)

def calculate_similarity_metrics(labels, similarity_matrix):
    intra_values = []
    inter_values = []

    n = len(labels)

    for i in range(n):
        for j in range(i + 1, n):
            sim = similarity_matrix[i, j]

            if np.isnan(sim):
                continue

            if labels[i] == labels[j]:
                intra_values.append(sim)
            else:
                inter_values.append(sim)

    intra_similarity = np.mean(intra_values) if len(intra_values) > 0 else np.nan
    inter_similarity = np.mean(inter_values) if len(inter_values) > 0 else np.nan
    similarity_gap = intra_similarity - inter_similarity

    return intra_similarity, inter_similarity, similarity_gap

results = []

for k in range(min_k, max_k + 1):
    print(f"\nEvaluating KMeans with k={k}...")

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(embeddings)

    silhouette = silhouette_score(embeddings, labels, metric="cosine")
    calinski = calinski_harabasz_score(embeddings, labels)
    davies = davies_bouldin_score(embeddings, labels)

    intra_similarity, inter_similarity, similarity_gap = calculate_similarity_metrics(
        labels,
        similarity_matrix
    )

    results.append({
        "k": k,
        "silhouette_score": silhouette,
        "calinski_harabasz_score": calinski,
        "davies_bouldin_score": davies,
        "intra_similarity": intra_similarity,
        "inter_similarity": inter_similarity,
        "similarity_gap": similarity_gap
    })

result_df = pd.DataFrame(results)

print("\n===== 클러스터 수별 평가 결과 =====")
display(result_df)

score_df = result_df.copy()

# 높을수록 좋은 지표
higher_better_cols = [
    "silhouette_score",
    "calinski_harabasz_score",
    "intra_similarity",
    "similarity_gap"
]

# 낮을수록 좋은 지표
lower_better_cols = [
    "davies_bouldin_score",
    "inter_similarity"
]

scaler = MinMaxScaler()

for col in higher_better_cols:
    score_df[col + "_scaled"] = scaler.fit_transform(score_df[[col]])

for col in lower_better_cols:
    scaled = scaler.fit_transform(score_df[[col]])
    score_df[col + "_scaled"] = 1 - scaled

scaled_cols = [col + "_scaled" for col in higher_better_cols + lower_better_cols]

score_df["total_score"] = score_df[scaled_cols].mean(axis=1)

best_row = score_df.loc[score_df["total_score"].idxmax()]
best_k = int(best_row["k"])

print("\n===== 최적 클러스터 수 =====")
print(f"Best k: {best_k}")
print(f"Total Score: {best_row['total_score']:.4f}")

best_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

best_labels = best_kmeans.fit_predict(embeddings)

df_valid["cluster"] = best_labels

df.loc[df_valid.index, "cluster"] = best_labels

final_intra, final_inter, final_gap = calculate_similarity_metrics(
    best_labels,
    similarity_matrix
)

cluster_results = []

for cluster_id in sorted(df_valid["cluster"].unique()):
    cluster_indices = np.where(best_labels == cluster_id)[0]

    if len(cluster_indices) < 2:
        cluster_intra = np.nan
    else:
        cluster_sim_matrix = similarity_matrix[np.ix_(cluster_indices, cluster_indices)]
        cluster_intra = np.nanmean(cluster_sim_matrix)

    cluster_results.append({
        "cluster": cluster_id,
        "num_documents": len(cluster_indices),
        "intra_similarity": cluster_intra
    })

cluster_result_df = pd.DataFrame(cluster_results)

print("\n===== 최종 Similarity 결과 =====")
print(f"Best k                    : {best_k}")
print(f"Intra-cluster Similarity  : {final_intra:.4f}")
print(f"Inter-cluster Similarity  : {final_inter:.4f}")
print(f"Similarity Gap            : {final_gap:.4f}")

print("\n===== 최종 클러스터별 Intra Similarity =====")
display(cluster_result_df)

with open(txt_filename, "w", encoding="utf-8") as f:
    f.write("문서 간 의미 유사도 비교 결과\n")
    f.write("=" * 60 + "\n\n")

    f.write("[사용 모델]\n")
    f.write("Embedding Model: intfloat/multilingual-e5-large-instruct\n")
    f.write(f"Embedding Column: {embedding_col}\n")
    f.write("Clustering Method: KMeans\n")
    f.write(f"Cluster Search Range: {min_k} ~ {max_k}\n")
    f.write(f"Number of Documents: {len(df_valid)}\n\n")

    f.write("[클러스터 수별 평가 결과]\n")
    f.write(result_df.to_string(index=False))
    f.write("\n\n")

    f.write("[종합 점수 기준 평가 결과]\n")
    f.write(score_df[["k", "total_score"] + scaled_cols].to_string(index=False))
    f.write("\n\n")

    f.write("[최적 클러스터 수]\n")
    f.write(f"Best k: {best_k}\n")
    f.write(f"Total Score: {best_row['total_score']:.4f}\n\n")

    f.write("[최적 k 기준 전체 Similarity 결과]\n")
    f.write(f"Intra-cluster Similarity : {final_intra:.4f}\n")
    f.write(f"Inter-cluster Similarity : {final_inter:.4f}\n")
    f.write(f"Similarity Gap           : {final_gap:.4f}\n\n")

print(f"\nTXT file saved to {txt_filename}")

사용 가능한 데이터 수: 10000
임베딩 차원: 768

Evaluating KMeans with k=2...

Evaluating KMeans with k=3...

Evaluating KMeans with k=4...

Evaluating KMeans with k=5...

Evaluating KMeans with k=6...

Evaluating KMeans with k=7...

Evaluating KMeans with k=8...

Evaluating KMeans with k=9...

Evaluating KMeans with k=10...

===== 클러스터 수별 평가 결과 =====


,k,silhouette_score,calinski_harabasz_score,davies_bouldin_score,intra_similarity,inter_similarity,similarity_gap
0,2,0.142868,734.340990,3.665077,0.409739,0.298877,0.110863
1,3,0.082135,538.702098,3.922496,0.429350,0.316442,0.112907
2,4,0.078670,446.964135,3.971150,0.459411,0.316129,0.143281
3,5,0.081228,383.503199,3.752592,0.469809,0.320185,0.149624
4,6,0.063328,341.250150,3.698085,0.470569,0.327267,0.143302
5,7,0.068603,315.114100,3.395658,0.475419,0.327047,0.148372
6,8,0.056860,291.294778,3.632721,0.472253,0.334053,0.138200
7,9,0.053102,274.251166,3.503090,0.478148,0.335409,0.142740
8,10,0.062567,262.524055,3.467200,0.494509,0.333892,0.160617



===== 최적 클러스터 수 =====
Best k: 2
Total Score: 0.5886

===== 최종 Similarity 결과 =====
Best k                    : 2
Intra-cluster Similarity  : 0.4097
Inter-cluster Similarity  : 0.2989
Similarity Gap            : 0.1109

===== 최종 클러스터별 Intra Similarity =====


,cluster,num_documents,intra_similarity
0,0,5443,0.506133
1,1,4557,0.272214



TXT file saved to /content/similarity_comparison_result_005.txt
